In [ ]:
# Install dependencies (run once)
!pip install -q transformers datasets "seqeval" spacy pandas scikit-learn streamlit pillow pytesseract nbformat
!python -m spacy download en_core_web_sm
print("Installed dependencies.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Installed dependencies.


In [ ]:
# Imports & setup
import re, os, json, textwrap
from typing import List, Dict, Tuple
import numpy as np, pandas as pd
import spacy
nlp_spacy = spacy.load("en_core_web_sm")

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
print("Imports ready.")

Imports ready.


In [ ]:
# Optional dataset loading (may fail if not present on HF hub)
from huggingface_hub import list_datasets
print("Total HF datasets available (sample):", len(list(list_datasets())) )

datasets = {}
try:
    datasets['ncbi_disease'] = load_dataset("ncbi_disease")
    print("Loaded NCBI Disease:", {k: len(v) for k,v in datasets['ncbi_disease'].items()})
except Exception as e:
    print("NCBI load failed:", e)

try:
    datasets['bc5cdr'] = load_dataset("bc5cdr")
    print("Loaded BC5CDR:", {k: len(v) for k,v in datasets['bc5cdr'].items()})
except Exception as e:
    print("BC5CDR load failed:", e)

print("\nNote: MIMIC-III and i2b2 require access and are not auto-downloadable.")

Total HF datasets available (sample): 582930
NCBI load failed: Dataset scripts are no longer supported, but found ncbi_disease.py
BC5CDR load failed: Dataset 'bc5cdr' doesn't exist on the Hub or cannot be accessed.

Note: MIMIC-III and i2b2 require access and are not auto-downloadable.


In [ ]:
# Preprocessing utilities
def clean_text(text: str) -> str:
    text = text.replace("\r", " ").replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def preprocess_text(text: str) -> str:
    text = clean_text(text)
    expansions = {r"\bBP\b":"blood pressure", r"\bHR\b":"heart rate", r"\bSOB\b":"shortness of breath", r"\bTID\b":"three times a day"}
    for pat,repl in expansions.items():
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    return text

sample_text = "Patient: 45-year-old male with fever and cough. Prescribed Paracetamol 500 mg twice daily."
print(preprocess_text(sample_text))

Patient: 45-year-old male with fever and cough. Prescribed Paracetamol 500 mg twice daily.


In [ ]:
# Load Hugging Face pipelines
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

NER_MODEL = "dmis-lab/biobert-base-cased-v1.1"
SUM_MODEL = "t5-small"

print("Loading NER pipeline...")
try:
    ner_pipe = pipeline("token-classification", model=NER_MODEL, aggregation_strategy="simple")
    print("Loaded BioBERT token-classification pipeline.")
except Exception as e:
    print("BioBERT token-classifier unavailable or lacks head. Falling back to generic NER. Error:", e)
    ner_pipe = pipeline("ner", aggregation_strategy="simple")
    print("Loaded generic NER pipeline.")

print("Loading summarizer...")
try:
    summ_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL)
    summ_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL)
    print("Loaded T5-small summarizer.")
except Exception as e:
    print("Summarizer load failed:", e)
    summ_tokenizer = None; summ_model = None

Loading NER pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (6 outputs left out)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded T5-small summarizer.


In [ ]:
# Entity extraction function
def extract_entities(text: str):
    text = preprocess_text(text)
    results = []
    try:
        ents = ner_pipe(text)
        for e in ents:
            ent_text = e.get("word") or e.get("entity") or ""
            ent_label = e.get("entity_group") or e.get("entity") or ""
            results.append({"text": ent_text, "label": ent_label, "score": float(e.get("score",0.0))})
    except Exception as e:
        print("NER error:", e)
    uniq = {}
    for r in results:
        k = r["text"].strip().lower()
        if not k: continue
        if k not in uniq or r.get("score",0.0) > uniq[k].get("score",0.0):
            uniq[k] = r
    return list(uniq.values())

print("Entities:", extract_entities(sample_text))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Entities: [{'text': 'patient : 45 -', 'label': 'LABEL_0', 'score': 0.6045485734939575}, {'text': 'year', 'label': 'LABEL_1', 'score': 0.5064879655838013}, {'text': '- old', 'label': 'LABEL_0', 'score': 0.5517079830169678}, {'text': 'male', 'label': 'LABEL_1', 'score': 0.5050818920135498}, {'text': 'with', 'label': 'LABEL_0', 'score': 0.5689917802810669}, {'text': 'fever', 'label': 'LABEL_1', 'score': 0.5403985381126404}, {'text': 'and cough. prescribed paracetamol 500', 'label': 'LABEL_0', 'score': 0.5916945338249207}, {'text': 'mg', 'label': 'LABEL_1', 'score': 0.5144228339195251}, {'text': 'twice daily.', 'label': 'LABEL_0', 'score': 0.5929385423660278}]


In [ ]:
# Summarization using T5-small
def summarize_text(text: str, max_len=100):
    text = preprocess_text(text)
    if summ_model is None:
        return "Summarizer not available."
    input_text = "summarize: " + text
    input_ids = summ_tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = summ_model.generate(input_ids, max_length=max_len, min_length=20, num_beams=4, early_stopping=True)
    return summ_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Summary:", summarize_text(sample_text, max_len=60))

Summary: Prescribed paracetamol 500 mg twice daily. patient: 45-year-old male with fever and cough.


In [ ]:
# Summarization using T5-small
def summarize_text(text: str, max_len=100):
    text = preprocess_text(text)
    if summ_model is None:
        return "Summarizer not available."
    input_text = "summarize: " + text
    input_ids = summ_tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = summ_model.generate(input_ids, max_length=max_len, min_length=20, num_beams=4, early_stopping=True)
    return summ_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Summary:", summarize_text(sample_text, max_len=60))

Summary: Prescribed paracetamol 500 mg twice daily. patient: 45-year-old male with fever and cough.


In [ ]:
# Placeholder for Relation Extraction
def extract_relations(text: str, entities: List[Dict]) -> List[Dict]:
    # This is a placeholder. A full implementation would involve
    # using a relation extraction model to find relationships between entities.
    return []

# Placeholder for Disease Prediction
def disease_predictor_from_entities(entities: List[Dict], text: str) -> str:
    # This is a placeholder. A full implementation would involve
    # analyzing entities and potentially external knowledge bases to predict diseases.
    diseases = [ent["text"] for ent in entities if ent["label"].lower() in ["disease", "symptom"]]
    if diseases: return f"Potential diseases/symptoms identified: {', '.join(diseases)}"
    return "No specific disease predicted based on entities."

# Full pipeline wrapper & demo
class MediNLP_Pipeline:
    def analyze(self, text: str):
        txt = preprocess_text(text)
        ents = extract_entities(txt)
        rels = extract_relations(txt, ents)
        summ = summarize_text(txt)
        disease = disease_predictor_from_entities(ents, txt)
        return {"text": txt, "entities": ents, "relations": rels, "summary": summ, "disease": disease}

demo_text = """
Patient: 45-year-old male presenting with fever, cough and sore throat for 3 days.
Prescribed: Paracetamol 500 mg twice daily and Azithromycin 500 mg once daily.
History of hypertension.
"""
pipe = MediNLP_Pipeline()
res = pipe.analyze(demo_text)
import pprint; pprint.pprint(res)

{'disease': 'No specific disease predicted based on entities.',
 'entities': [{'label': 'LABEL_0',
               'score': 0.606347382068634,
               'text': 'patient : 45 -'},
              {'label': 'LABEL_1', 'score': 0.5159802436828613, 'text': 'year'},
              {'label': 'LABEL_0',
               'score': 0.5478255748748779,
               'text': '- old'},
              {'label': 'LABEL_1', 'score': 0.5066609978675842, 'text': 'male'},
              {'label': 'LABEL_0',
               'score': 0.5528503656387329,
               'text': 'presenting with'},
              {'label': 'LABEL_1', 'score': 0.538622260093689, 'text': 'fever'},
              {'label': 'LABEL_0',
               'score': 0.5665741562843323,
               'text': ', cough and sore'},
              {'label': 'LABEL_1',
               'score': 0.5036307573318481,
               'text': 'throat'},
              {'label': 'LABEL_0',
               'score': 0.6084796190261841,
               'text': '

In [ ]:
# Fine-tuning template (commented)
# from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
# model_name = 'dmis-lab/biobert-base-cased-v1.1'
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=NUM_LABELS)
# # Prepare datasets in HF format with 'tokens' and 'labels'
# training_args = TrainingArguments(output_dir='./ner_finetuned', num_train_epochs=3, per_device_train_batch_size=8)
# trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset, tokenizer=tokenizer)
# trainer.train()
print('Fine-tuning template provided.')

Fine-tuning template provided.


In [ ]:
import streamlit as st
import tempfile, os

# The original line caused a ModuleNotFoundError as medinlp_pipeline.py does not exist.
# We will assume MediNLP_Pipeline is defined in a previous cell and is globally available.
# from medinlp_pipeline import MediNLP  # assumes medinlp_pipeline.py is present in same folder

st.set_page_config(page_title="MediNLP Demo", layout="centered")
st.title("MediNLP — Medical Text Analysis")
st.markdown("Paste medical text or upload an image (OCR) to extract entities, relations, summaries, and predictions.")

input_type = st.radio("Input Type", ["Text", "Image"])

if input_type == "Text":
    text = st.text_area("Paste medical text here", height=250)
    if st.button("Analyze Text"):
        with st.spinner("Analyzing..."):
            # Use the class name as defined in the notebook's global scope
            pipeline = MediNLP_Pipeline()
            # Correct method name to 'analyze' as defined in MediNLP_Pipeline
            res = pipeline.analyze(text)
            st.subheader("Extracted Entities")
            st.json(res["entities"])
            st.subheader("Relations")
            st.json(res["relations"])
            st.subheader("Summary")
            st.write(res["summary"])
            st.subheader("Predicted Disease")
            st.write(res["disease"])
else:
    uploaded_file = st.file_uploader("Upload image (jpg/png)", type=["png","jpg","jpeg"])
    if uploaded_file is not None:
        tfile = tempfile.NamedTemporaryFile(delete=False)
        tfile.write(uploaded_file.getvalue())
        tfile.flush()
        st.image(tfile.name, caption="Uploaded Image", use_column_width=True)
        if st.button("Extract & Analyze"):
            with st.spinner("Running OCR & NLP..."):
                st.warning("OCR functionality is not yet implemented in the `MediNLP_Pipeline` class.")
                # pipeline = MediNLP_Pipeline()
                # text = pipeline.ocr_image(tfile.name) # ocr_image is not defined in MediNLP_Pipeline
                # st.subheader("Extracted Text (OCR)")
                # st.write(text)
                # res = pipeline.analyze(text)
                # st.subheader("Entities")
                # st.json(res["entities"])
                # st.subheader("Relations")
                # st.json(res["relations"])
                # st.subheader("Summary")
                # st.write(res["summary"])
                # st.subheader("Predicted Disease")
                # st.write(res["disease"])
        try:
            os.unlink(tfile.name)
        except:
            pass

2025-12-11 08:28:02.566 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-11 08:28:02.569 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-11 08:28:03.308 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-11 08:28:03.311 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-11 08:28:03.314 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-11 08:28:03.320 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-11 08:28:03.326 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
"""
medinlp_pipeline.py
Core pipeline components for MediNLP prototype.
"""

import re, json
from typing import List, Dict, Tuple

# Simple preprocessing
def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.replace('\n',' ').replace('\r',' ')).strip()

def preprocess_text(text: str) -> str:
    text = clean_text(text)
    expansions = {r"\\bBP\\b":"blood pressure", r"\\bHR\\b":"heart rate"}
    for pat,repl in expansions.items():
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    return text

# Placeholder functions for pipeline demonstration
def extract_entities(text: str):
    # Dummy entity extraction
    return []

def extract_relations(text: str, entities: List[Dict]):
    return []

def summarize_text(text: str):
    return "Summary not available."

class MediNLP:
    def __init__(self):
        pass
    def analyze_text(self, text: str):
        txt = preprocess_text(text)
        ents = extract_entities(txt)
        rels = extract_relations(txt, ents)
        summ = summarize_text(txt)
        return {"text": txt, "entities": ents, "relations": rels, "summary": summ}

if __name__ == '__main__':
    print('medinlp_pipeline placeholder')


medinlp_pipeline placeholder


In [ ]:
transformers
spacy
pandas
scikit-learn
streamlit
pillow
pytesseract


In [ ]:
"""
medinlp_pipeline.py
Full pipeline implementation for MediNLP prototype:
 - preprocessing
 - NER (Hugging Face token-classification pipeline for BioBERT or fallback)
 - simple relation extraction heuristics
 - summarization (T5)
 - demo disease predictor
 - OCR helper using pytesseract
Provides MediNLP class with analyze_text method used by medinlp_app.py
"""
import re, json
from typing import List, Dict, Tuple

# transformers & pipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from PIL import Image
import pytesseract

# simple preprocessing
def clean_text(text: str) -> str:
    text = text.replace("\\r", " ").replace("\\n", " ").strip()
    text = re.sub(r"\\s+", " ", text)
    return text

def preprocess_text(text: str) -> str:
    text = clean_text(text)
    abbreviations = {
        r"\\bBP\\b": "blood pressure",
        r"\\bHR\\b": "heart rate",
        r"\\bSOB\\b": "shortness of breath",
        r"\\bTID\\b": "three times a day",
        r"\\bqhs\\b": "every night at bedtime"
    }
    for pat, repl in abbreviations.items():
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    return text

# Load pipelines when module imported
# Attempt to use a biomedical token-classification model; fall back to generic NER
DEFAULT_NER_MODEL = "dmis-lab/biobert-base-cased-v1.1"
try:
    ner_pipe = pipeline("token-classification", model=DEFAULT_NER_MODEL, aggregation_strategy="simple")
except Exception as e:
    # fallback
    ner_pipe = pipeline("ner", aggregation_strategy="simple")

# Summarizer
DEFAULT_SUM_MODEL = "t5-small"
try:
    summ_tokenizer = AutoTokenizer.from_pretrained(DEFAULT_SUM_MODEL)
    summ_model = AutoModelForSeq2SeqLM.from_pretrained(DEFAULT_SUM_MODEL)
except Exception as e:
    summ_tokenizer = None
    summ_model = None

# NER extractor class
class MediNLPNER:
    def __init__(self, ner_pipeline=None):
        self.ner = ner_pipeline or ner_pipe

    def extract_entities(self, text: str) -> List[Dict]:
        text = preprocess_text(text)
        results = []
        try:
            ents = self.ner(text)
            for e in ents:
                ent_text = e.get("word") or e.get("entity") or ""
                ent_label = e.get("entity_group") or e.get("entity") or ""
                results.append({"text": ent_text, "label": ent_label, "score": float(e.get("score",0.0))})
        except Exception as e:
            # return empty list on failure
            return []
        # dedupe on lowercase text, keep highest score
        uniq = {}
        for r in results:
            k = r["text"].strip().lower()
            if not k: continue
            if k not in uniq or r.get("score",0.0) > uniq[k].get("score",0.0):
                uniq[k] = r
        return list(uniq.values())

# Relation extraction heuristics
def extract_relations(text: str, entities: List[Dict]) -> List[Tuple[str,str,str]]:
    relations = []
    # dosage regex
    dosage_pattern = re.compile(r"\\b\\d+(?:\\.\\d+)?\\s*(mg|g|ml|mcg|units|tablet|tabs|caps)\\b", flags=re.I)
    for m in dosage_pattern.finditer(text):
        dosage = m.group(0)
        best_ent = None
        best_dist = 1e9
        for ent in entities:
            lbl = ent.get("label","").upper()
            if any(x in lbl for x in ["DRUG","CHEM","MED","RX","MEDICATION","MISC"]):
                pos = text.lower().find(ent["text"].lower())
                dist = abs(pos - m.start())
                if dist < best_dist:
                    best_dist = dist
                    best_ent = ent
        if best_ent:
            relations.append((best_ent["text"], "has_dosage", dosage))
    # symptom-disease sentence co-occurrence
    sentences = re.split(r'[.\\n!]+', text)
    syms = [e for e in entities if "SYM" in e.get("label","").upper() or "SIGN" in e.get("label","").upper() or "SYMPTOM" in e.get("label","").upper()]
    dis = [e for e in entities if "DIS" in e.get("label","").upper() or "DISEASE" in e.get("label","").upper()]
    for s in sentences:
        for sym in syms:
            if sym["text"].lower() in s.lower():
                for d in dis:
                    if d["text"].lower() in s.lower():
                        relations.append((sym["text"], "associated_with", d["text"]))
    return relations

# Summarization function
def summarize_text(text: str, max_length: int = 120) -> str:
    text = preprocess_text(text)
    if summ_model is None:
        return "Summarizer not available."
    input_text = "summarize: " + text
    input_ids = summ_tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = summ_model.generate(input_ids, max_length=max_length, min_length=20, num_beams=4, early_stopping=True)
    return summ_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Demo disease predictor
def disease_predictor_from_entities(entities: List[Dict], text: str) -> Dict:
    disease_entities = [e for e in entities if "DISEASE" in e.get("label","").upper() or "DIS" in e.get("label","").upper()]
    if disease_entities:
        best = max(disease_entities, key=lambda x: x.get("score",0))
        return {"prediction": best["text"], "confidence": float(best.get("score",0.5)), "method": "entity"}
    txt = text.lower()
    if ("fever" in txt and "cough" in txt) or "flu" in txt:
        return {"prediction": "flu", "confidence": 0.6, "method": "rule"}
    if ("chest pain" in txt) or ("shortness of breath" in txt):
        return {"prediction": "cardiac_issue", "confidence": 0.55, "method": "rule"}
    return {"prediction": "unknown", "confidence": 0.2, "method": "rule"}

# OCR helper
def ocr_image(image_path: str) -> str:
    try:
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text
    except Exception as e:
        return ""

# Full pipeline class
class MediNLP:
    def __init__(self):
        self.ner = MediNLPNER()
    def analyze_text(self, text: str) -> Dict:
        txt = preprocess_text(text)
        entities = self.ner.extract_entities(txt)
        relations = extract_relations(txt, entities)
        summary = summarize_text(txt)
        disease = disease_predictor_from_entities(entities, txt)
        return {"text": txt, "entities": entities, "relations": relations, "summary": summary, "disease": disease}

# If run as script, demo

# If run as script, demo
if __name__ == "__main__":
    demo = """Patient: 45-year-old male presenting with fever and cough for 3 days.
Prescribed Paracetamol 500 mg twice daily and Azithromycin 500 mg once daily.
"""
    m = MediNLP()
    import json
    print(json.dumps(m.analyze_text(demo), indent=2))


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{
  "text": "Patient: 45-year-old male presenting with fever and cough for 3 days. \nPrescribed Paracetamol 500 mg twice daily and Azithromycin 500 mg once daily.",
  "entities": [
    {
      "text": "patient : 45 - year -",
      "label": "LABEL_1",
      "score": 0.5884066820144653
    },
    {
      "text": "old",
      "label": "LABEL_0",
      "score": 0.5153627991676331
    },
    {
      "text": "male",
      "label": "LABEL_1",
      "score": 0.6312967538833618
    },
    {
      "text": "presenting",
      "label": "LABEL_0",
      "score": 0.5121764540672302
    },
    {
      "text": "with fever and cough",
      "label": "LABEL_1",
      "score": 0.5678598284721375
    },
    {
      "text": "for",
      "label": "LABEL_0",
      "score": 0.5579452514648438
    },
    {
      "text": "3 days. prescribed paracetamol 500",
      "label": "LABEL_1",
      "score": 0.5837097764015198
    },
    {
      "text": "mg",
      "label": "LABEL_0",
      "score": 0.542159914970398
  

In [35]:
# Install required packages (run once)
!pip install -q --upgrade transformers datasets seqeval accelerate wandb
!pip install -q "huggingface-hub"
!python -m pip install -q "evaluate"

# Mount Google Drive (so checkpoints persist)
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 7.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import os, re, random, json
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from typing import List, Dict

# evaluation
import evaluate
seqeval = evaluate.load("seqeval")


In [31]:
# Model config - change model_name to a Clinical/Bio BERT if you prefer
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"  # baseline; if this lacks token head we still fine-tune a fresh head
CHECKPOINT_DIR = "/content/drive/MyDrive/medinlp_ner_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
NUM_EPOCHS = 3
BATCH_SIZE = 8   # reduce if OOM
LEARNING_RATE = 5e-5
MAX_LENGTH = 128
